In [1]:
import datetime

import polars as pl
from torch.distributed.tensor import zeros

# Explore

In [2]:
acc = pl.read_csv('../data/HI-Medium_accounts.csv')
trans = pl.read_csv('../data/HI-Medium_Trans.csv').with_columns(
    pl.col("Timestamp").str.strptime(pl.Datetime, "%Y/%m/%d %H:%M")
)

FileNotFoundError: No such file or directory (os error 2): ../data/HI-Medium_accounts.csv

In [ ]:
acc.describe()

In [ ]:
acc.head(10)

In [ ]:
trans.describe()

In [ ]:
trans.head(10)

In [52]:
accounts = trans.select(pl.col('From')).vstack(trans.select(pl.col('To')).rename({'To': 'From'})).unique().rename({'From': 'Number'}).with_row_index()
accounts.describe()

statistic,index,Number
str,f64,str
"""count""",2.076999e6,"""2076999"""
"""null_count""",0.0,"""0"""
"""mean""",1.038499e6,null
"""std""",599578.110216,null
"""min""",0.0,"""100428660"""
"""25%""",519250.0,null
"""50%""",1.038499e6,null
"""75%""",1.557749e6,null
"""max""",2.076998e6,"""8521FC1B0"""


In [53]:
accounts.head()

index,Number
u32,str
0,"""841891780"""
1,"""845E0E0F0"""
2,"""80BBE4C40"""
3,"""82A0DA9C0"""
4,"""832517F40"""


In [54]:
trans.filter(pl.col('Amount Paid') != pl.col('Amount Received')).head()

Timestamp,From Bank,From,To Bank,To,Amount Received,Receiving Currency,Amount Paid,Payment Currency,Payment Format,Is Laundering
datetime[μs],i64,str,i64,str,f64,str,f64,str,str,i64
2022-09-01 00:19:00,4011,"""8032D1A00""",4011,"""8032D1A00""",97.27,"""Euro""",113.98,"""US Dollar""","""ACH""",0
2022-09-01 00:13:00,5991,"""80341B8B0""",5991,"""80341B8B0""",14.26,"""UK Pound""",18.42,"""US Dollar""","""ACH""",0
2022-09-01 00:18:00,0,"""800417500""",0,"""800417500""",42.41,"""Euro""",49.69,"""US Dollar""","""ACH""",0
2022-09-01 00:11:00,3504,"""800492D00""",3504,"""800492D00""",6707.83,"""US Dollar""",5724.46,"""Euro""","""ACH""",0
2022-09-01 00:29:00,1488,"""800C948C0""",1488,"""800C948C0""",42.71,"""Euro""",50.05,"""US Dollar""","""ACH""",0


In [55]:
import gc

del accounts
del acc
del trans

gc.collect()

586

# Medium

In [56]:
trans = (
    pl.read_csv('../data/HI-Medium_Trans.csv')
    .with_columns(
        pl.col("Timestamp").str.strptime(pl.Datetime, "%Y/%m/%d %H:%M")
    )
    .sort('Timestamp')
)

# Cutoff for time leakage

In [57]:
import datetime

## Time splits for row proportion splitting

In [58]:
zero = trans['Timestamp'].min()
sixty = trans['Timestamp'].quantile(0.6)
eighty = trans['Timestamp'].quantile(0.8)
hundred = trans['Timestamp'].max()

print(zero, sixty, eighty, hundred)

2022-09-01 00:00:00 2022-09-09 20:43:00 2022-09-14 05:46:00 2022-09-28 15:58:00


In [59]:
train_row =  trans.filter(pl.col('Timestamp') <= sixty)
val_row = trans.filter(pl.col('Timestamp') <= eighty).filter(pl.col('Timestamp') > sixty)
test_row = trans.filter(pl.col('Timestamp') > eighty)

## Time splits for day splitting

In [60]:
diff = hundred - zero
days = diff.days

In [61]:
sixty = zero + datetime.timedelta(days=days * 0.6)
eighty = zero + datetime.timedelta(days=days * 0.8)
hundred = zero + datetime.timedelta(days=days)

print(zero, sixty, eighty, hundred)

2022-09-01 00:00:00 2022-09-17 04:48:00 2022-09-22 14:24:00 2022-09-28 00:00:00


In [62]:
train =  trans.filter(pl.col('Timestamp') <= sixty)
val = trans.filter(pl.col('Timestamp') <= eighty).filter(pl.col('Timestamp') > sixty)
test = trans.filter(pl.col('Timestamp') > eighty)

# Currency behavior

In [63]:
trans.group_by(pl.col('From')).agg(pl.len()).sort(pl.col('len'), descending=True).limit(10)

From,len
str,u32
"""100428660""",1076979
"""1004286A8""",678929
"""1004286F0""",208695
"""1004289C0""",132783
"""100428858""",102358
"""100428810""",96007
"""1004287C8""",90963
"""1004288A0""",87870
"""100428738""",80578


In [64]:
trans.filter(pl.col('From').is_in(["100428660", "1004286A8", "1004286F0"])).group_by(['From', 'Receiving Currency']).len()

From,Receiving Currency,len
str,str,u32
"""1004286A8""","""Euro""",678929
"""1004286F0""","""Yuan""",208695
"""100428660""","""US Dollar""",1076979


In [65]:
trans.filter(pl.col('From').is_in(["100428660", "1004286A8", "1004286F0"])).group_by(['From', 'Payment Currency']).len()

From,Payment Currency,len
str,str,u32
"""100428660""","""US Dollar""",1076979
"""1004286A8""","""Euro""",678929
"""1004286F0""","""Yuan""",208695
